# BHJet OpenMP zone benchmark

This notebook tests the `experiment/openmp-zones` branch. It verifies that the local experimental extension is imported, checks the total and component interfaces, then benchmarks the same BHJet solve with one and four OpenMP threads.

Run it from the OpenMP worktree with the `gammapy-2.1` kernel. The benchmark starts clean child Python processes, so it can control `OMP_NUM_THREADS` reliably.

In [ ]:
from pathlib import Path
import os
import sys


def find_project_root(start):
    for directory in (start, *start.parents):
        if (directory / 'python' / 'bhjet').is_dir() and (directory / 'CMakeLists.txt').is_file():
            return directory
    raise RuntimeError('Run this notebook from inside the modBHJet OpenMP worktree.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PYTHON_SOURCE = PROJECT_ROOT / 'python'
sys.path.insert(0, str(PYTHON_SOURCE))

print(f'Project root: {PROJECT_ROOT}')
print(f'Python source: {PYTHON_SOURCE}')

In [ ]:
import bhjet

print(f'BHJet package: {bhjet.__file__}')
assert str(PROJECT_ROOT) in bhjet.__file__, 'The notebook is not using the OpenMP worktree.'

## Total spectrum and components

The first total-model call computes a BHJet solution. `evaluate_components()` immediately afterward should reuse that solution and return unit-preserving `E dN/dE` quantities.

In [ ]:
import astropy.units as u
import numpy as np
import time

from bhjet.gammapy import BHJetSpectralModel

energy = np.geomspace(1e-8, 1e12, 201) * u.eV
model = BHJetSpectralModel()

start = time.perf_counter()
total_dnde = model(energy)
total_elapsed = time.perf_counter() - start

start = time.perf_counter()
components = model.evaluate_components(energy)
components_elapsed = time.perf_counter() - start

expected = {'pre_syn', 'pre_com', 'post_syn', 'post_com', 'total'}
assert set(components) == expected
assert total_dnde.shape == energy.shape
assert total_dnde.unit.is_equivalent('cm-2 s-1 erg-1')

for name, ednde in components.items():
    assert ednde.shape == energy.shape
    assert ednde.unit.is_equivalent('cm-2 s-1')
    print(f'{name:8s}: {ednde.unit}')

print(f'First total evaluation: {total_elapsed:.3f} s')
print(f'Component extraction:   {components_elapsed:.3f} s')

## One versus four OpenMP threads

Each child process builds a fresh model, so its timing includes exactly one complete BHJet calculation. The returned spectra must agree before interpreting any speed-up.

In [ ]:
import json
import subprocess


BENCHMARK_CODE = r'''
import json
import time
import astropy.units as u
import numpy as np
from bhjet.gammapy import BHJetSpectralModel

energy = np.geomspace(1e-8, 1e12, 201) * u.eV
model = BHJetSpectralModel()
start = time.perf_counter()
total = model(energy)
elapsed = time.perf_counter() - start
components = model.evaluate_components(energy)

print(json.dumps({
    'elapsed_s': elapsed,
    'dnde_total': total.to_value('cm-2 s-1 erg-1').tolist(),
    **{name: values.to_value('cm-2 s-1').tolist() for name, values in components.items()},
}))
'''


def run_benchmark(threads):
    environment = os.environ.copy()
    environment['OMP_NUM_THREADS'] = str(threads)
    environment['PYTHONPATH'] = str(PYTHON_SOURCE) + os.pathsep + environment.get('PYTHONPATH', '')

    result = subprocess.run(
        [sys.executable, '-c', BENCHMARK_CODE],
        env=environment,
        check=True,
        capture_output=True,
        text=True,
    )
    return json.loads(result.stdout.splitlines()[-1])


one_thread = run_benchmark(1)
four_threads = run_benchmark(4)

for name in expected:
    np.testing.assert_allclose(one_thread[name], four_threads[name], rtol=1e-12, atol=0.0)

speedup = one_thread['elapsed_s'] / four_threads['elapsed_s']
print(f"1 thread:  {one_thread['elapsed_s']:.3f} s")
print(f"4 threads: {four_threads['elapsed_s']:.3f} s")
print(f"Speed-up:   {speedup:.2f}x")
print('All total and component arrays agree at rtol=1e-12.')

## Optional: Gammapy integration

This cell loads the NGC 4261 radio, X-ray, and gamma-ray flux-point datasets and evaluates their combined statistic. Run it after the core benchmark.

In [ ]:
from astropy.table import Table
from gammapy.datasets import Datasets, FluxPointsDataset
from gammapy.estimators import FluxPoints
from gammapy.modeling.models import Models


def load_isis_fluxpoints(filename):
    data = np.atleast_2d(np.loadtxt(filename))
    nu_min = data[:, 0] * u.Hz
    nu_max = data[:, 1] * u.Hz
    e_min = nu_min.to(u.eV, equivalencies=u.spectral())
    e_ref = np.sqrt(nu_min * nu_max).to(u.eV, equivalencies=u.spectral())
    e_max = nu_max.to(u.eV, equivalencies=u.spectral())
    e_ref_erg = e_ref.to(u.erg)
    flux = data[:, 2] * u.Unit('erg cm-2 s-1')
    flux_error = 0.5 * (data[:, 3] + data[:, 4]) * u.Unit('erg cm-2 s-1')

    table = Table()
    table['e_min'] = e_min
    table['e_ref'] = e_ref
    table['e_max'] = e_max
    table['dnde'] = (flux / e_ref_erg**2).to('cm-2 s-1 eV-1')
    table['dnde_err'] = (flux_error / e_ref_erg**2).to('cm-2 s-1 eV-1')
    return FluxPoints.from_table(table, sed_type='dnde')


data_root = PROJECT_ROOT / 'examples' / 'gammapy' / 'data' / 'ngc4261'
radio_table = Table.read(data_root / 'ngc4261_dnde.csv', format='ascii.csv', delimiter=' ')
radio_table.sort('e_ref')
radio_table['e_ref'].unit = u.eV
radio_table['dnde'].unit = u.Unit('cm-2 s-1 eV-1')
radio_table['dnde_err'].unit = u.Unit('cm-2 s-1 eV-1')

datasets = Datasets([
    FluxPointsDataset(data=FluxPoints.from_table(radio_table, sed_type='dnde'), name='radio'),
    FluxPointsDataset(data=load_isis_fluxpoints(data_root / 'ngc4261_xray.txt'), name='xray'),
    FluxPointsDataset(data=load_isis_fluxpoints(data_root / 'ngc4261_gamma_ray.txt'), name='gamma'),
])
datasets.models = Models.read(data_root / 'ngc4261_test.yaml')

start = time.perf_counter()
statistic = datasets.stat_sum()
print(f'Combined statistic: {statistic:.12f}')
print(f'Elapsed time:       {time.perf_counter() - start:.3f} s')